In [ ]:
pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os

BASE_URL = "https://books.toscrape.com/"

rating_map = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
}

GBP_TO_INR = 105.50

books = []


def get_category_links():
    response = requests.get(BASE_URL)
    soup = BeautifulSoup(response.text,"html.parser")

    category_section = soup.find("ul",class_="nav nav-list")
    categories = category_section.find_all("a")[1:4]      # First 3 categories

    links=[]

    for c in categories:
        name=c.text.strip()
        href=BASE_URL+c["href"]
        links.append((name,href))

    return links


def scrape_category(category,url):

    while True:

        response=requests.get(url)
        soup=BeautifulSoup(response.text,"html.parser")

        articles=soup.find_all("article",class_="product_pod")

        for article in articles:

            title=article.h3.a["title"]

            price_text=article.find("p",class_="price_color").text

            rating_text=article.p["class"][1]

            availability=article.find("p",class_="instock availability").text.strip()

            books.append({
                "title":title,
                "price":price_text,
                "star_rating":rating_text,
                "availability":availability,
                "category":category
            })

        next_button=soup.find("li",class_="next")

        if next_button:

            next_url=url.rsplit("/",1)[0]+"/"+next_button.a["href"]
            url=next_url

        else:
            break


for cat,url in get_category_links():
    scrape_category(cat,url)

df=pd.DataFrame(books)

# ---------------- Cleaning ----------------

df["price_gbp"]=(
    df["price"]
    .str.replace("£","",regex=False)
    .str.replace("Â","",regex=False)
    .astype(float)
)

df["rating"]=df["star_rating"].map(rating_map)

df["in_stock"]=df["availability"].str.contains("In stock")

# Handle parsing failures

df["rating"].fillna(df["rating"].median(),inplace=True)

df["price_gbp"].fillna(df["price_gbp"].median(),inplace=True)

df["price_inr"]=round(df["price_gbp"]*GBP_TO_INR,2)

os.makedirs('outputs', exist_ok=True)

df.to_csv("outputs/cleaned_books.csv",index=False)

print(df.head())

print("Total Books =",len(df))

                                               title    price star_rating  \
0                            It's Only the Himalayas  Â£45.17         Two   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43        Four   
2  See America: A Celebration of Our National Par...  Â£48.87       Three   
3  Vagabonding: An Uncommon Guide to the Art of L...  Â£36.94         Two   
4                               Under the Tuscan Sun  Â£37.33       Three   

  availability category  price_gbp  rating  in_stock  price_inr  
0     In stock   Travel      45.17       2      True    4765.44  
1     In stock   Travel      49.43       4      True    5214.86  
2     In stock   Travel      48.87       3      True    5155.78  
3     In stock   Travel      36.94       2      True    3897.17  
4     In stock   Travel      37.33       3      True    3938.31  
Total Books = 69


/tmp/ipykernel_1261/2134274615.py:97: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["rating"].fillna(df["rating"].median(),inplace=True)
/tmp/ipykernel_1261/2134274615.py:99: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [ ]:
import sqlite3
import pandas as pd

df=pd.read_csv("outputs/cleaned_books.csv")

conn=sqlite3.connect("books.db")

cursor=conn.cursor()

cursor.execute("""

CREATE TABLE IF NOT EXISTS categories(

category_id INTEGER PRIMARY KEY AUTOINCREMENT,

category_name TEXT UNIQUE

)

""")

cursor.execute("""

CREATE TABLE IF NOT EXISTS books(

book_id INTEGER PRIMARY KEY AUTOINCREMENT,

title TEXT,

price_gbp REAL,

price_inr REAL,

rating INTEGER,

in_stock INTEGER,

category_id INTEGER,

FOREIGN KEY(category_id)

REFERENCES categories(category_id)

)

""")

# Insert Categories

cats=df["category"].unique()

for c in cats:

    cursor.execute(

        "INSERT OR IGNORE INTO categories(category_name) VALUES(?)",(c,)

    )

conn.commit()

cat_df=pd.read_sql("SELECT * FROM categories",conn)

mapping=dict(zip(cat_df.category_name,cat_df.category_id))

df["category_id"]=df["category"].map(mapping)

book_df=df[[
"title",
"price_gbp",
"price_inr",
"rating",
"in_stock",
"category_id"
]]

book_df.to_sql(
"books",
conn,
if_exists="append",
index=False
)

conn.commit()

print("Database Created Successfully")

conn.close()

Database Created Successfully


In [ ]:
import sqlite3
import pandas as pd

conn=sqlite3.connect("books.db")

queries={

"Q1":

"""
SELECT title,price_gbp
FROM books
WHERE rating>=4;
""",

"Q2":

"""
SELECT title,price_gbp
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
""",

"Q3":

"""
SELECT DISTINCT rating
FROM books;
""",

"Q4":

"""
SELECT title,price_gbp
FROM books
WHERE price_gbp
BETWEEN 20 AND 40;
""",

"Q5":

"""
SELECT
b.title,
c.category_name,
b.rating,
b.price_gbp

FROM books b

JOIN categories c

ON b.category_id=c.category_id

ORDER BY c.category_name,b.rating DESC

LIMIT 10;

"""
}

for name,query in queries.items():

    print("\n",name)

    result=pd.read_sql(query,conn)

    print(result)

    result.to_csv(f"outputs/{name}.csv",index=False)

# read_sql examples

df1=pd.read_sql(queries["Q2"],conn)

df2=pd.read_sql(queries["Q5"],conn)

print(df1.head())

print(df2.head())

# merge equivalent

books=pd.read_sql("SELECT * FROM books",conn)

cats=pd.read_sql("SELECT * FROM categories",conn)

merged=pd.merge(

books,

cats,

on="category_id"

)

merged=merged[[
"title",
"category_name",
"rating",
"price_gbp"
]]

print(merged.head())

merged.to_csv("outputs/merge_output.csv",index=False)

conn.close()


 Q1
                                                title  price_gbp
0   Full Moon over Noahâs Ark: An Odyssey to Mou...      49.43
1                    A Year in Provence (Provence #1)      56.88
2                  1,000 Places to See Before You Die      26.08
3                                       Sharp Objects      47.82
4                                 The Past Never Ends      56.50
5     The Murder of Roger Ackroyd (Hercule Poirot #4)      44.10
6              A Time of Torment (Charlie Parker #14)      48.35
7   Murder at the 42nd Street Library (Raymond Amb...      54.36
8   What Happened on Beale Street (Secrets of the ...      25.37
9   The Bachelor Girl's Guide to Murder (Herringfo...      52.30
10   Delivering the Truth (Quaker Midwife Mystery #1)      20.89
11  The Mysterious Affair at Styles (Hercule Poiro...      24.80
12                  The Silkworm (Cormoran Strike #2)      23.05
13  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70
14                  

Required SQL Queries Covered

In [ ]:
| Query    | Requirement |

| WHERE    | Yes           |
| ORDER BY | Yes           |
| LIMIT    | Yes           |
| DISTINCT | Yes           |
| BETWEEN  | Yes           |
| JOIN     | Yes           |

Cleaning Decision

In [ ]:
price_gbp -> Median Imputation

rating -> Median Imputation